# MAGI — Colab GPU Benchmark

Runs the MAGI training pipeline on a Colab GPU runtime and times it against the
CPU baselines recorded in the repo's own notebooks, to see how much a CUDA GPU
actually speeds up training. Two parts:

1. **Part A — synthetic-data sanity check.** Self-contained, no data upload,
   mirrors `Example_Usage.ipynb`. Confirms the GPU path runs correctly
   end-to-end before committing to the long real-data run.
2. **Part B — real-data benchmark ("CR" source).** Mirrors
   `MAGI_v0_8_2.ipynb`'s 40-epoch / batch-4096 training run, so wall-clock is
   directly comparable to that notebook's own comment: **~48 min on an M1
   CPU**. Needs the "CR" training file uploaded to Google Drive first (see
   Part B below for the exact path).

**Before running anything:** in the Colab menu, go to
*Runtime → Change runtime type → Hardware accelerator → GPU* (T4 is fine),
then *Runtime → Restart session* if you already ran cells under the wrong
runtime.

`francescomonastra/MAGI` is a **private** repo, so the clone cell below needs
a GitHub Personal Access Token, not just the URL:

1. Go to <https://github.com/settings/tokens?type=beta> → *Generate new
   token* → give it access to only the `MAGI` repository, *Contents:
   Read-only* permission is enough.
2. Copy the token (starts with `github_pat_...`).
3. Run the next cell — it will prompt for the token with a hidden input
   (via `getpass`), so it is never shown, printed, or saved in the notebook.

This notebook is meant to run **on Colab**, not locally — it clones the repo
fresh and does not touch any local MAGI files.


In [ ]:
import getpass
import os

REPO_DIR = "MAGI"

if not os.path.isdir(REPO_DIR):
    # MAGI is a private repo: a plain `git clone https://...` has no
    # credentials to offer in a non-interactive Colab shell and fails with
    # "could not read Username". Paste a GitHub Personal Access Token
    # instead (see the markdown cell above for how to generate one) - getpass
    # keeps it out of the cell's input/output, and it is discarded right
    # after the clone.
    token = getpass.getpass("GitHub Personal Access Token: ")
    clone_url = f"https://{token}@github.com/francescomonastra/MAGI.git"
    !git clone -q {clone_url} {REPO_DIR}
    del token, clone_url

%cd {REPO_DIR}
!pip install -q -e MAGI_package/

# `pip install -e` registers the package via a .pth file that Python's site
# module only reads at interpreter startup - so it is NOT importable in this
# already-running kernel yet, even though the install just succeeded. Put the
# package directory on sys.path directly instead of restarting the runtime.
import sys
sys.path.insert(0, os.path.abspath("MAGI_package"))

import magi
print("magi loaded from:", magi.__file__)


In [ ]:
# Check BEFORE importing magi: if this shows no GPU, fix the runtime type now
# (Runtime > Change runtime type > GPU) rather than after a long install/train.
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Physical devices  :", tf.config.list_physical_devices())
print("GPU devices       :", tf.config.list_physical_devices("GPU"))

assert tf.config.list_physical_devices("GPU"), (
    "No GPU visible to TensorFlow. In Colab: Runtime > Change runtime type > "
    "Hardware accelerator > GPU, then Runtime > Restart session and re-run "
    "from the top."
)


In [ ]:
# Unlike Example_Usage.ipynb / MAGI_v0_8_2.ipynb, do NOT set
# CUDA_VISIBLE_DEVICES=-1 and pass cpu_only=False: that is the entire change
# needed for magi to use the GPU.
import time

import numpy as np
import pandas as pd
import magi

magi.initialize_environment(seed=42, cpu_only=False, quiet=True)
magi.print_tf_info()


## Part A — synthetic-data sanity check

Same synthetic source and pipeline as `Example_Usage.ipynb`: a toy 7,200-event
spectrum with two spectral lines and an injected energy↔direction correlation,
small enough to train in seconds/minutes. No data upload needed. The numbers
this produces are not meaningful physics — the point is to prove the GPU path
runs correctly before committing to the long real-data run in Part B.


In [ ]:
rng = np.random.default_rng(42)

N_CONT = 6000            # continuum events
N_LINE_A, N_LINE_B = 700, 500
E_LINE_A = 0.00800571    # MeV — a Cu K-alpha1-like fluorescence line
E_LINE_B = 0.510999      # MeV — e+e- annihilation

center = (0.0, 0.0, -50.0)   # mm
R = 20.0                     # mm

# --- energies: power-law continuum spanning ~4 decades, plus two exact lines
u = rng.uniform(0.0, 1.0, N_CONT)
E_cont = 1e-3 * (1.0 / (1.0 - 0.999 * u)) ** 0.6          # MeV, 1e-3 .. ~1e1
E_cont = np.clip(E_cont, 1e-3, 20.0)

E = np.concatenate([
    E_cont,
    np.full(N_LINE_A, E_LINE_A),      # exactly monoenergetic
    np.full(N_LINE_B, E_LINE_B),
])
N = E.size
order = rng.permutation(N)
E = E[order]

# --- particle type: energy-dependent, so the type conditioning has real work
p_gamma = np.clip(0.7 - 0.15 * np.log10(E / 1e-3), 0.05, 0.9)
which = rng.uniform(size=N)
ParticleName = np.where(which < p_gamma, "gamma",
                np.where(which < p_gamma + 0.25, "e-", "mu-"))

# --- position: uniform on the sphere surface (global-frame u_r/phi_r).
u_r = rng.uniform(-1.0, 1.0, N)
phi_r = rng.uniform(0.0, 2 * np.pi, N)
sin_r = np.sqrt(1.0 - u_r ** 2)
X = center[0] + R * sin_r * np.cos(phi_r)
Y = center[1] + R * sin_r * np.sin(phi_r)
Z = center[2] + R * u_r

# --- direction: correlated with energy (global-frame u_v/phi_v).
forward = np.clip(0.15 + 0.30 * np.log10(E / 1e-3), 0.0, 1.0)
u_v = np.clip(rng.normal(loc=-forward, scale=0.35, size=N), -0.999, 0.999)
phi_v = rng.uniform(0.0, 2 * np.pi, N)
sin_v = np.sqrt(1.0 - u_v ** 2)

Vx = sin_v * np.cos(phi_v)
Vy = sin_v * np.sin(phi_v)
Vz = u_v                      # by construction, this is MAGI's u_v

print("injected corr(log10 E, u_v) =",
      round(float(np.corrcoef(np.log10(E), u_v)[0, 1]), 3))

df = pd.DataFrame({
    "ParticleName": ParticleName,
    "Energy": E,
    "X": X, "Y": Y, "Z": Z,
    "Vx": Vx, "Vy": Vy, "Vz": Vz,
    "PrimBool": rng.integers(0, 2, N),
})

magi.report_basic_table_checks(df)


In [ ]:
prep = magi.build_physical_features(
    df, center=center, radius=R, source_type="cosmic_ray")
magi.print_physical_summary(prep)

candidate_lines = [
    {"label": "synthetic Cu K-alpha1", "energy_mev": E_LINE_A, "origin": "instrumental"},
    {"label": "synthetic e+e- 511",    "energy_mev": E_LINE_B, "origin": "instrumental"},
]

RESOLUTION_EV = 4.0   # X-IFU-like; the GENERATIVE line width

E_raw = prep["features"]["Energy"].to_numpy()

res = magi.detect_energy_lines(
    E_raw,
    binning_mode="log_fixed_count", n_bins=256,
    prominence_factor=3.0, window=5,
    candidate_lines=candidate_lines,
    refine_bin_width_mev=RESOLUTION_EV * 1e-6,
)
magi.print_detected_energy_lines(res)

matched = [m for m in res["matched_lines"] if m["count"] >= 100]
matched += magi.confirm_unresolved_candidate_lines(
    E_raw, candidate_lines, matched, resolution_ev=RESOLUTION_EV)

print("\nmodelled lines:", [m["label"] for m in matched])


In [ ]:
feature_pack = magi.build_feature_dataframe(
    prep,
    energy_binning_mode="log_fixed_count",
    n_bins=64,                 # small, because this toy source is small
    min_counts=5,
    geometry_transform="quantile_u_r_u_v_phi_r_phi_v",
    n_quantiles=1000,
    random_state=42,
    energy_transform="log10",
)
magi.report_feature_dataframe(feature_pack)

E_full = feature_pack["filtered_prep"]["features"]["Energy"].to_numpy()

gate_targets = magi.build_gate_targets(
    E_full, feature_pack["energy_bins"], matched,
    bandwidth_mode="resolution",
    bandwidth_fwhm_mev=RESOLUTION_EV * 1e-6,
)

line_positions_y = np.log10(
    [m["candidate_energy_mev"] for m in matched]).astype(np.float32)
line_logsigma = magi.line_logsigma_from_resolution(
    10.0 ** line_positions_y, RESOLUTION_EV, fwhm=True)

feat = feature_pack["feat"].copy()
for j in range(gate_targets.shape[1]):
    feat[f"gate_target_{j}"] = gate_targets[:, j]

cont_cols = ("u_r_q", "u_v_q", "phi_r_q", "phi_v_q", "energy_y") + tuple(
    f"gate_target_{j}" for j in range(gate_targets.shape[1]))

dataset_pack = magi.filter_particle_types_continuous_geometry(
    feat=feat, prob_threshold=1e-5, cont_cols=cont_cols,
    normalization=prep.get("normalization"))

split_pack = magi.split_feature_data(dataset_pack, random_state=42)
magi.report_split_summary(split_pack, dataset_pack["n_types"])

scaled_pack = magi.scale_continuous_features(split_pack, scale_cols=())
condition_pack = magi.build_conditioning_and_weights(
    scaled_pack, dataset_pack["n_types"],
    idx_to_type=dataset_pack["idx_to_type"], alpha=0.5)
magi.report_conditioning(condition_pack, dataset_pack["n_types"])

ds_pack = magi.build_tf_datasets(condition_pack, batch_size=256)
magi.report_tf_datasets(ds_pack)

# Per-type zone probabilities for prior_zone_conditioning (see MAGI_v0_8_2.ipynb).
n_zones = gate_targets.shape[1]
zone_cols = dataset_pack["X_cont_raw"][:, -n_zones:]
y_type = dataset_pack["y_type"]

zone_probs = np.zeros((dataset_pack["n_types"], n_zones), dtype=np.float64)
for t in range(dataset_pack["n_types"]):
    mask = (y_type == t)
    row = zone_cols[mask].mean(axis=0) if mask.any() else np.zeros(n_zones)
    s = row.sum()
    zone_probs[t] = (row / s) if s > 0 else np.eye(n_zones)[0]


In [ ]:
ey = feat["energy_y"].to_numpy()
warp_yk, warp_zk = magi.fit_cdf_warp_knots(ey, n_knots=64, eps=1e-4)

model_a = magi.CVAE_MixEnergy_ContPhi_TaskAdaptive(
    n_types=dataset_pack["n_types"],
    latent_dim=4,
    hidden=(64, 64),
    line_positions_y=line_positions_y,
    line_logsigma_init=line_logsigma,
    line_logsigma_trainable=False,   # widths are pinned, not fitted
    continuum_mode="flow",
    continuum_flow_warp="cdf",
    continuum_flow_warp_y_knots=warp_yk,
    continuum_flow_warp_z_knots=warp_zk,
    continuum_flow_bins=16,
    continuum_flow_transforms=3,
    energy_flow_condition="z_cond",
    prior="coupling",
    prior_zone_conditioning=True,
    zone_probs=zone_probs,
    gate_focal_gamma=1.0,
    w_gate_aux=2.0,
    type_weights=condition_pack["type_weights"],
)

magi.compile_model(model_a, learning_rate=2e-3, optimizer="adam")


In [ ]:
t0 = time.time()

history_a = magi.fit_model(
    model_a, ds_pack["train_ds"], ds_pack["val_ds"],
    epochs=8, callbacks=magi.build_default_callbacks(early_patience=20),
    verbose=2,
)

gpu_time_synthetic_s = time.time() - t0
print(f"\nPart A (synthetic, 8 epochs) GPU wall-clock: {gpu_time_synthetic_s:.1f}s")
print('Reference: Example_Usage.ipynb runs the same 8 epochs '
      '"in a couple of minutes" on CPU.')


In [ ]:
# Quick check that GPU training reached a sane model, not just a fast one:
# same coupling-residual / Wasserstein checks as Example_Usage.ipynb section 12.
f_real = feature_pack["filtered_prep"]["features"]
real_a = {
    "E": f_real["Energy"].to_numpy(),
    "logE": np.log10(f_real["Energy"].to_numpy()),
    "u_r": f_real["u_r"].to_numpy(),
    "u_v": f_real["u_v"].to_numpy(),
    "phi_r": f_real["phi_r"].to_numpy(),
    "phi_v": f_real["phi_v"].to_numpy(),
}

gen_pack_a = magi.generate_latent_outputs(
    model_a, n_samples=real_a["E"].size,
    type_probs=dataset_pack["type_probs"],
    n_types=dataset_pack["n_types"],
    idx_to_type=dataset_pack["idx_to_type"],
    rng=np.random.default_rng(0),
)

qt_a = feature_pack["quantile_transformers"]
reco_a = magi.reconstruct_generated_features(
    gen_pack_a,
    energy_head_mode="mixture",
    energy_transform="log10",
    geometry_mode="quantile_u_r_u_v_phi_r_phi_v",
    qt_u_r=qt_a["qt_u_r"], qt_u_v=qt_a["qt_u_v"],
    qt_phi_r=qt_a["qt_phi_r"], qt_phi_v=qt_a["qt_phi_v"],
)
gen_a = {
    "E": reco_a["E_gen"], "logE": np.log10(reco_a["E_gen"]),
    "u_r": reco_a["u_r_gen"], "u_v": reco_a["u_v_gen"],
    "phi_r": reco_a["phi_r_gen"], "phi_v": reco_a["phi_v_gen"],
}

from scipy.stats import wasserstein_distance

for k in ["logE", "u_r", "u_v", "phi_r", "phi_v"]:
    print(f"  {k:8s} Wasserstein = {wasserstein_distance(real_a[k], gen_a[k]):.4f}")

cols = ["logE", "u_r", "u_v", "phi_r", "phi_v"]
corr_real_a = pd.DataFrame({c: real_a[c] for c in cols}).corr()
corr_gen_a = pd.DataFrame({c: gen_a[c] for c in cols}).corr()
residual_a = (corr_gen_a - corr_real_a).abs().to_numpy().max()
print(f"\ncoupling residual max|d corr| = {residual_a:.3f}  "
      "(not meaningful at this sample size/epoch count - see Example_Usage.ipynb)")


## Part B — real-data benchmark ("CR" source)

Mirrors `MAGI_v0_8_2.ipynb`'s actual training run (40 epochs, batch size 4096,
same model config) so the GPU wall-clock below is directly comparable to that
notebook's own comment: **~48 min on an M1 CPU** for the "CR" source.

**Before running this part**, upload `alloutputDSCryoSphereCR.dat` (246 MB,
gitignored — not included in the clone) to your Google Drive at:

```
MyDrive/MAGI_data/alloutputDSCryoSphereCR.dat
```

The candidate-lines JSON is already tracked in git, so it came with the clone
in Part A — no upload needed for it.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TRAINING_DATA_DIR = "/content/drive/MyDrive/MAGI_data"
SOURCE = "CR"
SOURCE_FILE = f"{TRAINING_DATA_DIR}/alloutputDSCryoSphereCR.dat"

# Already present from the git clone in Part A - no upload needed.
CANDIDATE_LINES_FILE = (
    "CandidateLines/CANDIDATE_ENERGY_LINES_SRON_CCNwithXFDM_NoShield_"
    "FlowerCryoAC_fixed_EADL.json"
)

assert os.path.isfile(SOURCE_FILE), (
    f"{SOURCE_FILE} not found - upload alloutputDSCryoSphereCR.dat to "
    f"{TRAINING_DATA_DIR}/ in your Google Drive first."
)


In [ ]:
center_b = (0.0, 0.0, -507.66)
R_b = 100.0
X_IFU_RESOLUTION_EV = 4.0   # X-IFU energy resolution (FWHM), pins the line widths

df_b = magi.load_detector_table(filepath=SOURCE_FILE, sep=r"\s+")
magi.report_basic_table_checks(df_b)

prep_b = magi.build_physical_features(df_b, center=center_b, radius=R_b)
magi.print_physical_summary(prep_b)

E_all_b = prep_b["features"]["Energy"].to_numpy()
print(f"\n{SOURCE}: {len(df_b):,} rows, {E_all_b.size:,} valid energies")

candidate_payload = magi.load_candidate_energy_lines(CANDIDATE_LINES_FILE)
candidate_lines_b = candidate_payload["lines"]
print(f"Loaded {candidate_payload['n_lines']} candidate lines "
      f"for mass model: {candidate_payload['mass_model']}")

line_result_b = magi.detect_energy_lines(
    E_all_b,
    binning_mode="log_fixed_count",
    n_bins=1024,
    prominence_factor=3.0,
    window=5,
    candidate_lines=candidate_lines_b,
    refine_bin_width_mev=X_IFU_RESOLUTION_EV * 1e-6,
)
magi.print_detected_energy_lines(line_result_b)

matched_all_b = line_result_b["matched_lines"]
matched_b = [m for m in matched_all_b if m["count"] >= 100]
print(f"\n{len(matched_all_b)} matched lines; {len(matched_b)} feed the mixture "
      f"head (count>=100): {[m['label'] for m in matched_b]}")


In [ ]:
feature_pack_b = magi.build_feature_dataframe(
    prep_b,
    energy_binning_mode="log_fixed_count",
    n_bins=512,
    geometry_transform="quantile_u_r_u_v_phi_r_phi_v",
    n_quantiles=10000,
    random_state=42,
    energy_transform="log10",
)
magi.report_feature_dataframe(feature_pack_b)

line_positions_mev_b = np.array(
    [m["candidate_energy_mev"] for m in matched_b], dtype=np.float64)
line_positions_y_b = np.log10(line_positions_mev_b).astype(np.float32)

E_full_b = feature_pack_b["filtered_prep"]["features"]["Energy"].to_numpy()
FWHM_MEV_B = X_IFU_RESOLUTION_EV * 1e-6

gate_targets_b = magi.build_gate_targets(
    E_full_b, feature_pack_b["energy_bins"], matched_b,
    bandwidth_mode="resolution",          # confirmed default, see MAGI_v0_8_2.ipynb
    bandwidth_fwhm_mev=FWHM_MEV_B,
)

feat_b = feature_pack_b["feat"].copy()
for j in range(gate_targets_b.shape[1]):
    feat_b[f"gate_target_{j}"] = gate_targets_b[:, j]

cont_cols_b = ("u_r_q", "u_v_q", "phi_r_q", "phi_v_q", "energy_y") + tuple(
    f"gate_target_{j}" for j in range(gate_targets_b.shape[1]))

dataset_pack_b = magi.filter_particle_types_continuous_geometry(
    feat=feat_b, prob_threshold=1e-5, cont_cols=cont_cols_b,
)
magi.report_continuous_geometry_features(dataset_pack_b)

# Per-type zone probabilities for prior_zone_conditioning (v0.8.2).
n_zones_b = gate_targets_b.shape[1]
zone_cols_b = dataset_pack_b["X_cont_raw"][:, -n_zones_b:]
y_type_b = dataset_pack_b["y_type"]

zone_probs_b = np.zeros((dataset_pack_b["n_types"], n_zones_b), dtype=np.float64)
for t in range(dataset_pack_b["n_types"]):
    mask = (y_type_b == t)
    row = zone_cols_b[mask].mean(axis=0) if mask.any() else np.zeros(n_zones_b)
    row_sum = row.sum()
    zone_probs_b[t] = (row / row_sum) if row_sum > 0 else np.eye(n_zones_b)[0]

split_pack_b = magi.split_feature_data(
    dataset_pack_b, test_size_total=0.30, val_size_from_temp=0.50, random_state=42)
magi.report_split_summary(split_pack_b, n_types=dataset_pack_b["n_types"])

scaled_pack_b = magi.scale_continuous_features(split_pack_b, scale_cols=())
condition_pack_b = magi.build_conditioning_and_weights(
    scaled_pack_b, n_types=dataset_pack_b["n_types"],
    idx_to_type=dataset_pack_b["idx_to_type"], alpha=0.5)

tf_pack_b = magi.build_tf_datasets(
    condition_pack_b, batch_size=4096, shuffle_buffer_cap=200_000)
magi.report_tf_datasets(tf_pack_b)

quantile_transformers_b = feature_pack_b["quantile_transformers"]
energy_y_all_b = feat_b["energy_y"].to_numpy()
warp_y_knots_b, warp_z_knots_b = magi.fit_cdf_warp_knots(
    energy_y_all_b, n_knots=256, eps=1e-4)


In [ ]:
# Same config as MAGI_v0_8_2.ipynb, so the GPU/CPU wall-clock comparison is
# apples-to-apples.
EPOCHS_B = 40
LEARNING_RATE_B = 2e-4

line_logsigma_init_b = magi.line_logsigma_from_resolution(
    line_positions_mev_b, X_IFU_RESOLUTION_EV, fwhm=True)

model_config_b = {
    "model_class": "CVAE_MixEnergy_ContPhi_TaskAdaptive",
    "n_types": dataset_pack_b["n_types"],
    "line_positions_y": line_positions_y_b.tolist(),
    "latent_dim": 8,
    "hidden": [128, 128, 64],
    "beta": 0.2,
    "continuum_mode": "flow",
    "continuum_flow_bins": 24,
    "continuum_flow_transforms": 3,
    "continuum_flow_warp": "cdf",
    "energy_flow_condition": "z_cond",
    "prior": "coupling",
    "w_gate_aux": 2.0,
    "gate_focal_gamma": 1.0,
    "gate_class_weights": None,
    "line_logsigma_trainable": False,
    "x_ifu_resolution_ev": X_IFU_RESOLUTION_EV,
    "prior_zone_conditioning": True,
    "zone_probs": zone_probs_b.tolist(),
}

model_b = magi.CVAE_MixEnergy_ContPhi_TaskAdaptive(
    n_types=model_config_b["n_types"],
    line_positions_y=line_positions_y_b,
    latent_dim=model_config_b["latent_dim"],
    hidden=tuple(model_config_b["hidden"]),
    beta=model_config_b["beta"],
    continuum_mode="flow",
    continuum_flow_bins=model_config_b["continuum_flow_bins"],
    continuum_flow_transforms=model_config_b["continuum_flow_transforms"],
    continuum_flow_warp="cdf",
    continuum_flow_warp_y_knots=warp_y_knots_b,
    continuum_flow_warp_z_knots=warp_z_knots_b,
    energy_flow_condition="z_cond",
    prior="coupling",
    w_gate_aux=model_config_b["w_gate_aux"],
    gate_focal_gamma=model_config_b["gate_focal_gamma"],
    gate_class_weights=model_config_b["gate_class_weights"],
    line_logsigma_init=line_logsigma_init_b,
    line_logsigma_trainable=False,
    prior_zone_conditioning=model_config_b["prior_zone_conditioning"],
    zone_probs=zone_probs_b,
)

magi.compile_model(model_b, learning_rate=LEARNING_RATE_B)

callbacks_b = magi.build_default_callbacks(
    monitor="val_loss", early_patience=8, lr_patience=6,
    factor=0.5, min_lr=1e-5, verbose=1,
)

magi.print_model_structure(model_b)


In [ ]:
# CPU reference (MAGI_v0_8_2.ipynb, same config): ~48 min for CR at 40 epochs.
t0 = time.time()

history_b = magi.fit_model(
    model=model_b,
    train_ds=tf_pack_b["train_ds"],
    val_ds=tf_pack_b["val_ds"],
    epochs=EPOCHS_B,
    callbacks=callbacks_b,
    verbose=2,
)

gpu_time_real_s = time.time() - t0
n_epochs_run_b = len(history_b.history["loss"])
print(f"\n{SOURCE}: trained {n_epochs_run_b} epochs in {gpu_time_real_s:.0f}s "
      f"({gpu_time_real_s/60:.1f} min) on GPU")
print("CPU reference (MAGI_v0_8_2.ipynb, same config): ~48 min for CR, 40 epochs.")


In [ ]:
# Sanity check, not a full acceptance run (see tools/acceptance_v0_8.py for
# that): does the GPU-trained model reach comparable accuracy, not just speed?
qt_b = quantile_transformers_b
X_cont_test_b = tf_pack_b["X_cont_test"]

NGEN_B = int(len(X_cont_test_b))
CHUNK_B = 500_000

def generate_physics_b(model, n_gen, chunk=CHUNK_B):
    parts = []
    done = 0
    while done < n_gen:
        m = min(chunk, n_gen - done)
        gen_raw = magi.generate_latent_outputs(
            model=model, n_samples=m,
            type_probs=dataset_pack_b["type_probs"],
            n_types=dataset_pack_b["n_types"],
            idx_to_type=dataset_pack_b["idx_to_type"],
        )
        gen_feat = magi.reconstruct_generated_features(
            gen_raw,
            energy_head_mode="mixture",
            energy_transform="log10",
            geometry_mode="quantile_u_r_u_v_phi_r_phi_v",
            qt_u_r=qt_b["qt_u_r"], qt_u_v=qt_b["qt_u_v"],
            qt_phi_r=qt_b["qt_phi_r"], qt_phi_v=qt_b["qt_phi_v"],
        )
        parts.append(magi.reconstruct_generated_physics(gen_feat, center=center_b, radius=R_b))
        done += m
    if len(parts) == 1:
        return parts[0]
    out = {}
    for k, v in parts[0].items():
        if isinstance(v, np.ndarray) and v.ndim >= 1 and v.shape[0] == parts[0]["E_gen"].shape[0]:
            out[k] = np.concatenate([p[k] for p in parts], axis=0)
        else:
            out[k] = v
    return out

gen_phys_b = generate_physics_b(model_b, NGEN_B)

E_test_raw_b = feature_pack_b["filtered_prep"]["features"]["Energy"].to_numpy()[
    split_pack_b["idx_test"]]

real_phys_b = magi.reconstruct_real_test_physics(
    X_cont_test=X_cont_test_b[:, :4],
    E_test_raw=E_test_raw_b,
    qt_u_r=qt_b["qt_u_r"], qt_u_v=qt_b["qt_u_v"],
    qt_phi_r=qt_b["qt_phi_r"], qt_phi_v=qt_b["qt_phi_v"],
    center=center_b, radius=R_b,
    geometry_mode="quantile_u_r_u_v_phi_r_phi_v",
)

magi.report_generated_constraints(gen_phys_b, radius=R_b)

scores_b = magi.compute_wasserstein_scores(real_phys_b, gen_phys_b)
print("\nWasserstein distances (real vs generated):")
for k, v in scores_b.items():
    print(f"  {k:8s} {v:.5f}")
print("  (bar on real data, per README.md v0.8.2 table: logE <= 0.05)")


In [ ]:
import joblib

SAVE_DIR_B = f"/content/drive/MyDrive/MAGI_data/trained_models/v0_8_2_{SOURCE}_colab_gpu"
MODEL_NAME_B = f"mix_{SOURCE}"

paths_b = magi.save_final_trained_model(
    model=model_b,
    save_dir=SAVE_DIR_B,
    model_name=MODEL_NAME_B,
    history=history_b,
    model_config=model_b.to_generation_config(),
    preprocessing_metadata={
        "source": SOURCE,
        "geometry_transform": "quantile_u_r_u_v_phi_r_phi_v",
        "energy_transform": "log10",
        "cont_cols": list(cont_cols_b),
        "energy_bins": [float(b) for b in feature_pack_b["energy_bins"]],
        "type_probs": np.asarray(dataset_pack_b["type_probs"]).tolist(),
        "idx_to_type": dataset_pack_b["idx_to_type"],
        "n_types": int(dataset_pack_b["n_types"]),
    },
    notes="MAGI_Colab_GPU_Benchmark.ipynb GPU run",
)

joblib.dump(
    quantile_transformers_b,
    os.path.join(SAVE_DIR_B, f"{MODEL_NAME_B}_quantile_transformers.joblib"),
)
print("\nsaved to:", SAVE_DIR_B)


## Summary

In [ ]:
print(f"{'run':30s} {'GPU wall-clock':>18s} {'CPU reference':>22s}")
print(f"{'Part A (synthetic, 8 ep)':30s} {str(round(gpu_time_synthetic_s, 1)) + 's':>18s} "
      f"{'~a couple of min (M1 CPU)':>22s}")
print(f"{'Part B (' + SOURCE + ', 40 ep)':30s} {str(round(gpu_time_real_s / 60, 1)) + 'm':>18s} "
      f"{'~48 min (M1 CPU)':>22s}")
